## CS 363M: Machine Learning Project
The goal of this project is to predict whether an aircraft wildlife strike resulted in damage to the aircraft (`INDICATED_DAMAGE = 1`) or not (`INDICATED_DAMAGE = 0`).

## Our Approach
Our work is organized as following:
1. Load and inspect the data
2. Identify missing values and data quality issues
3. Drop columns that are too sparse, redundant, or difficult to use meaningfully
4. Clean and impute remaining values
5. Encode categorical variables
6. Train and compare models
7. Evaluate performance and iterate

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## Load the Data

We begin by loading the training and test datasets. During our initial inspection, we noticed that missing values were recorded inconsistently — some rows used `"UNKNOWN"`, others `"Unknown"` or `"UNK"`. We standardize all of these to `NaN` at load time so that pandas treats them uniformly as missing, which gives us a more accurate picture of data quality before we start cleaning.

In [ ]:
# Load data, treating common unknown strings as NaN
train_data = pd.read_csv("data/train.csv", skipinitialspace=True, low_memory=False, na_values=["UNKNOWN", "Unknown", "UNK"])
test_data = pd.read_csv("data/test.csv", skipinitialspace=True, low_memory=False, na_values=["UNKNOWN", "Unknown", "UNK"])

# Save test IDs before we drop the column later
test_ids = test_data["INDEX_NR"].copy()

TARGET = "INDICATED_DAMAGE"

print("Percent Null:", train_data.isnull().sum().sum() / (len(train_data) * len(train_data.columns)) * 100)
print(train_data.isnull().sum())

print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)

## Initial Data Inspection and Cleaning

After reviewing the columns, we identified several that would not help the model make predictions. Some were administrative fields like `SOURCE` and `PERSON` that describe how the record was filed rather than what happened during the strike. Others like `REMARKS` and `COMMENTS` contain free-text notes that are too unstructured to use directly. `REG` and `FLT` identify a specific aircraft or flight, which makes them too granular to generalize to new data. Finally, `BIRD_BAND_NUMBER` was almost entirely empty since the vast majority of birds involved in strikes are never banded. We drop all of these before moving forward.

In [ ]:
def drop_irrelevant_columns(data):
    data = data.copy()
    drop_cols = ["INDEX_NR", "REMARKS", "COMMENTS", "PERSON", "SOURCE", "LUPDATE", "TRANSFER", "BIRD_BAND_NUMBER", "REG", "FLT"]
    
    # Dropping columns
    for col in drop_cols:
        if col in data.columns:
            data = data.drop(columns=[col])
            
    return data

train_data = drop_irrelevant_columns(train_data)
test_data = drop_irrelevant_columns(test_data)

## Feature Engineering

### Date Features

The `INCIDENT_DATE` column stores dates as strings, so we first convert it to a proper datetime format. From there we can extract the year, month, day of week, and quarter. For example, wildlife activity varies significantly by season, so we also create four binary season flags.

In [ ]:
def add_date_features(data):
    if "INCIDENT_DATE" not in data.columns:
        return data

    # Convert the string column to an actual datetime type so we can extract parts from it
    dates = pd.to_datetime(data["INCIDENT_DATE"], format="%m/%d/%y", errors="coerce")

    data["year"] = dates.dt.year
    data["month"] = dates.dt.month
    data["dayofweek"] = dates.dt.dayofweek
    data["quarter"] = dates.dt.quarter

    # Season flags
    data["is_spring"] = data["month"].isin([3, 4, 5]).astype(int)
    data["is_summer"] = data["month"].isin([6, 7, 8]).astype(int)
    data["is_fall"] = data["month"].isin([9, 10, 11]).astype(int)
    data["is_winter"] = data["month"].isin([12, 1, 2]).astype(int)

    data = data.drop(columns=["INCIDENT_DATE"])
    return data

### Numeric Column Conversion

Several columns that represent numeric quantities (like altitude, speed, and engine position) were stored as mixed-type or string columns in the raw CSV. If we leave them as strings, the model cannot use them numerically. We force each one to a proper numeric type, and any value that cannot be converted becomes `NaN`, which will be handled later by the imputer in our preprocessing pipeline.

In [ ]:
def convert_numeric_columns(data):
    # These columns can sometimes be read as strings, so we force them to numbers
    numeric_cols = [
        "HEIGHT", "SPEED", "DISTANCE", "NUM_SEEN", "NUM_STRUCK",
        "AC_MASS", "NUM_ENGS", "LATITUDE", "LONGITUDE",
        "AMA", "AMO", "EMA", "EMO",
        "ENG_1_POS", "ENG_2_POS", "ENG_3_POS", "ENG_4_POS"
    ]
    for col in numeric_cols:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")  # non-numeric values become NaN
    return data

### Bird Size Features

The `SIZE` column describes the bird as Small, Medium, or Large, but the raw string label is not directly usable by tree-based models in a meaningful ordinal way. We create an ordinal encoding (1, 2, 3) so the model understands that Large > Medium > Small in terms of potential impact. We also add two binary flags for large and medium birds so the model can learn non-linear relationships for specific size categories independently.

In [ ]:
def add_size_features(data):
    if "SIZE" not in data.columns:
        return data

    # Ordinal encoding so the model knows Large > Medium > Small
    size_map = {"Small": 1, "Medium": 2, "Large": 3}
    data["SIZE_ENCODED"] = data["SIZE"].map(size_map)
    data["is_large_bird"] = (data["SIZE"] == "Large").astype(int)
    data["is_medium_bird"] = (data["SIZE"] == "Medium").astype(int)
    return data

### Flight Phase Features

The phase of flight at the time of the strike turned out to be an important signal. Strikes during takeoff, climb, landing roll, and approach tend to be more severe because the aircraft is at lower altitudes, traveling at speeds where evasion is impossible, and the engines are under higher load. We flag each of these individually and also create a combined `is_critical_phase` feature to give the model a single high-level indicator of dangerous conditions.

In [ ]:
def add_flight_phase_features(data):
    if "PHASE_OF_FLIGHT" not in data.columns:
        return data

    data["is_takeoff"] = (data["PHASE_OF_FLIGHT"] == "Take-off Run").astype(int)
    data["is_landing"] = data["PHASE_OF_FLIGHT"].isin(["Landing Roll", "Approach"]).astype(int)
    data["is_climb"] = (data["PHASE_OF_FLIGHT"] == "Climb").astype(int)

    # Combined flag for all high-risk phases
    critical_phases = ["Take-off Run", "Landing Roll", "Approach", "Climb"]
    data["is_critical_phase"] = data["PHASE_OF_FLIGHT"].isin(critical_phases).astype(int)
    return data

### Time of Day Features

Strikes that happen at night or during dawn and dusk are more likely to result in damage because pilots have reduced visibility and less time to react. We create binary flags for these low-visibility periods so the model can learn this pattern directly rather than trying to extract it from a raw categorical string.

In [ ]:
def add_time_of_day_features(data):
    if "TIME_OF_DAY" not in data.columns:
        return data

    data["is_night"] = (data["TIME_OF_DAY"] == "Night").astype(int)
    data["is_dawn_dusk"] = data["TIME_OF_DAY"].isin(["Dawn", "Dusk"]).astype(int)
    return data

### Speed, Height, and Impact Features

Speed and altitude at the time of the strike are two of the most physically meaningful predictors. A bird struck at high speed carries far more kinetic energy, and strikes at low altitude give the crew no time to respond. We create an interaction term (`speed_height`) that captures the combined effect of both, and a `low_altitude` flag for strikes at or below 500 feet. We also combine speed with bird size to create a proxy for impact force, since a large bird at high speed is a very different scenario than a small bird at low speed.

In [ ]:
def add_speed_height_features(data):
    if "HEIGHT" not in data.columns or "SPEED" not in data.columns:
        return data

    data["speed_height"] = data["SPEED"] * data["HEIGHT"]
    data["low_altitude"] = (data["HEIGHT"] <= 500).astype(int)

    # Impact risk proxy: faster speed + larger bird = more damage potential
    if "SIZE_ENCODED" in data.columns:
        data["bird_aircraft_risk"] = data["SPEED"] * data["SIZE_ENCODED"]
    return data

### Strike Count Features

The dataset records both how many birds were seen and how many were actually struck. Rather than using these raw counts alone, we compute the ratio of birds struck to birds seen.

In [ ]:
def add_strike_count_features(data):
    if "NUM_SEEN" not in data.columns or "NUM_STRUCK" not in data.columns:
        return data

    data["strike_density"] = data["NUM_STRUCK"] / (data["NUM_SEEN"] + 1)
    return data

### Aircraft Mass Features

Heavier aircraft are generally more structurally robust, but they also have larger engines that birds can be ingested into. The `AC_MASS` column uses a categorical mass code rather than an exact weight. We flag aircraft in mass category 4 or above as heavy, which roughly corresponds to large commercial jets. This gives the model a simple way to distinguish commercial airliners from smaller regional aircraft.

In [ ]:
def add_aircraft_features(data):
    if "AC_MASS" not in data.columns:
        return data

    # Mass category 4+ corresponds to large commercial aircraft
    data["is_heavy_aircraft"] = (data["AC_MASS"] >= 4).astype(int)
    return data

### Weather Features

Adverse weather conditions can affect both the likelihood and severity of a strike. Rain or other precipitation reduces visibility, and overcast skies can push birds lower where aircraft traffic is heavier. We create a `has_precip` flag by checking whether the precipitation field has a real recorded value.

In [ ]:
def add_weather_features(data):
    if "PRECIPITATION" in data.columns:
        has_value = data["PRECIPITATION"].notna()
        no_precip_labels = ["None", "No", "Missing"]
        is_not_none = data["PRECIPITATION"].isin(no_precip_labels) == False
        data["has_precip"] = (has_value & is_not_none).astype(int)

    if "SKY" in data.columns:
        data["is_overcast"] = (data["SKY"] == "Overcast").astype(int)
        data["has_clouds"] = data["SKY"].isin(["Some Cloud", "Overcast"]).astype(int)
    return data

### Species Features

The species name is a free-text field with hundreds of distinct values, so too many to one-hot encode directly. Instead, we group species into risk categories based on keywords in the name. Raptors and large waterfowl like geese tend to cause more damage due to their size and flight patterns. Small songbirds are generally lower risk.

In [ ]:
def species_contains_any(species_series, keywords):
    # Start with all zeros, then set 1 wherever any keyword matches
    result = pd.Series([0] * len(species_series), index=species_series.index)
    for keyword in keywords:
        result[species_series.str.contains(keyword)] = 1
    return result


def add_species_features(data):
    if "SPECIES" not in data.columns:
        return data

    species = data["SPECIES"].fillna("").str.lower()

    raptor_keywords = ["hawk", "eagle", "falcon", "kestrel", "osprey", "owl", "vulture"]
    data["is_raptor"] = species_contains_any(species, raptor_keywords)

    water_bird_keywords = ["gull", "goose", "duck", "heron", "egret", "crane", "pelican", "cormorant"]
    data["is_water_bird"] = species_contains_any(species, water_bird_keywords)

    data["is_unknown_bird"] = species_contains_any(species, ["unknown"])

    songbird_keywords = ["swallow", "sparrow", "starling", "lark", "dove", "pigeon"]
    data["is_small_songbird"] = species_contains_any(species, songbird_keywords)

    large_bird_keywords = ["hawk", "eagle", "vulture", "goose", "pelican"]
    data["is_large_bird_group"] = species_contains_any(species, large_bird_keywords)

    return data

### Warning Features

The `WARNED` column records whether the flight crew received a warning about wildlife before the strike. We suspected that a warning with no resulting avoidance might correlate with more severe outcomes, and that an unwarned strike might reflect a surprise collision. We encode both the yes and no cases as separate binary features so the model can distinguish them from missing values, which indicate the information was simply not recorded.

In [ ]:
def add_warned_features(data):
    if "WARNED" not in data.columns:
        return data

    data["warned_yes"] = (data["WARNED"] == "Yes").astype(int)
    data["warned_no"] = (data["WARNED"] == "No").astype(int)
    return data

### Running Feature Engineering

With all helper functions defined, `make_features` acts as the single entry point that runs each transformation in sequence. We apply it to both the train and test sets so that both end up with the same feature columns.

In [ ]:
def make_features(data):
    data = data.copy()

    data = add_date_features(data)
    data = convert_numeric_columns(data)
    data = add_size_features(data)
    data = add_flight_phase_features(data)
    data = add_time_of_day_features(data)
    data = add_speed_height_features(data)
    data = add_strike_count_features(data)
    data = add_aircraft_features(data)
    data = add_weather_features(data)
    data = add_species_features(data)
    data = add_warned_features(data)

    return data


train_fe = make_features(train_data)
test_fe = make_features(test_data)

## Model Pipeline and Training

With our features ready, we set up the preprocessing pipeline and train two models: XGBoost and LightGBM. Both are gradient boosting algorithms that handle tabular data well. We use a `ColumnTransformer` to apply different preprocessing to numeric and categorical columns — median imputation for numeric, mode imputation plus one-hot encoding for categorical. Rare category values (appearing fewer than 15 times) are dropped from the one-hot encoding to avoid overfitting on noise.

Because damage cases are less common than non-damage cases, we also pass a class imbalance weight to each model so it does not simply learn to predict "no damage" for everything. We then use 5-fold stratified cross-validation and collect out-of-fold predictions, which lets us blend and evaluate the models honestly without touching the test set.

In [ ]:
y = train_fe[TARGET].astype(int)
X = train_fe.drop(columns=[TARGET])

# Align test set to the same columns as train
test_fe = test_fe.reindex(columns=X.columns, fill_value=np.nan)

numeric_cols = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()

numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])

categorical_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")),
                                          ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=15))])

preprocessor = ColumnTransformer(transformers=[("num", numeric_transformer, numeric_cols), ("cat", categorical_transformer, categorical_cols)], remainder="drop")

spw = (y == 0).sum() / (y == 1).sum()

xgb_clf = XGBClassifier(
    n_estimators=900,
    max_depth=7,
    learning_rate=0.03,
    subsample=0.90,
    colsample_bytree=0.80,
    min_child_weight=3,
    gamma=0.15,
    reg_alpha=0.25,
    reg_lambda=1.25,
    scale_pos_weight=spw,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

lgb_clf = LGBMClassifier(
    n_estimators=900,
    learning_rate=0.03,
    num_leaves=64,
    max_depth=8,
    min_child_samples=20,
    subsample=0.90,
    colsample_bytree=0.80,
    reg_alpha=0.25,
    reg_lambda=1.25,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

models = [("xgb", Pipeline(steps=[("preprocess", preprocessor), ("model", xgb_clf)])), 
          ("lgb", Pipeline(steps=[("preprocess", preprocessor), ("model", lgb_clf)]))]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros((len(X), len(models)))
test_proba = np.zeros((len(test_fe), len(models)))

for model_index, (name, model) in enumerate(models):
    print(f"\nTraining {name}")
    fold_test_probs = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train_fold = X.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        model.fit(X_train_fold, y_train_fold)

        val_probs = model.predict_proba(X_val_fold)[:, 1]
        test_probs = model.predict_proba(test_fe)[:, 1]

        oof[val_idx, model_index] = val_probs
        fold_test_probs.append(test_probs)

        fold_score = balanced_accuracy_score(y_val_fold, (val_probs >= 0.5).astype(int))
        print(f"  Fold {fold + 1}: {fold_score:.4f}")

    # Average test predictions across all folds for a more stable estimate
    test_proba[:, model_index] = np.mean(fold_test_probs, axis=0)

## Optimization

Rather than treating both models equally, we search for the best blend weight between XGBoost and LightGBM using their out-of-fold predictions. At the same time, we search for the best probability threshold for classifying a strike as damaging. The default threshold of 0.5 is rarely optimal on imbalanced datasets — tuning it on the OOF predictions lets us improve balanced accuracy without touching the test set.

In [ ]:
best_score = 0
best_weight = 0.5
best_thresh = 0.5

for w in np.arange(0.0, 1.01, 0.05):
    blended_oof = w * oof[:, 0] + (1 - w) * oof[:, 1]

    for t in np.arange(0.10, 0.90, 0.01):
        predictions = (blended_oof >= t).astype(int)
        score = balanced_accuracy_score(y, predictions)

        if score > best_score:
            best_score = score
            best_thresh = t
            best_weight = w

print(f"Best OOF Balanced Accuracy: {best_score:.4f}")
print(f"Best threshold: {best_thresh:.2f}")
print(f"Best XGB weight: {best_weight:.2f}")

## Final Prediction

Using our optimized parameters, we generated the final predictions for the test set. This ensemble approach provides a more robust and accurate prediction than our initial baseline.

In [ ]:
final_proba = best_weight * test_proba[:, 0] + (1 - best_weight) * test_proba[:, 1]
final_preds = (final_proba >= best_thresh).astype(int)

submission = pd.DataFrame({"INDEX_NR": test_ids.values, "INDICATED_DAMAGE": final_preds})

submission.to_csv("submission.csv", index=False)

print("Final submission created successfully.")